# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [1445]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [1446]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [1447]:
def build_classifiers():
    lr = LinearRegression()
    lr.fit(data_set, labels)

    knn = KNeighborsClassifier()
    knn.fit(data_set, labels)

    svc = SVC()
    svc.fit(data_set, labels)

    dt = DecisionTreeClassifier()
    dt.fit(data_set, labels)

    nb = GaussianNB()
    nb.fit(data_set, labels)

    qda = QuadraticDiscriminantAnalysis()
    qda.fit(data_set, labels)

    all = [lr, knn, svc, dt, nb, qda]
    return zip(*map(lambda x: all[x:], range(3)))

In [1448]:
def build_stacked_classifier(classifiers, stacked_classifier):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130, len(classifiers)))
    
    # stacked classifier part:
    stacked_classifier.fit(output.reshape((130, len(classifiers))), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]), len(classifiers)))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [1449]:
print_results = False

results = []
for stacked_classifier in (
        LinearRegression,
        KNeighborsClassifier,
        SVC,
        DecisionTreeClassifier,
        GaussianNB,
        QuadraticDiscriminantAnalysis
    ):
    sc = stacked_classifier()
    if print_results: print(f"{sc}: ")
    try:
        sc.fit(data_set, labels)
        predicted = sc.predict(test_data_set)
        accuracy = accuracy_score(test_labels, predicted)
        if print_results: print(f"\t{accuracy} alone")
        results.append((accuracy, f"{sc} alone"))
    except Exception as e:
        if print_results: print(f"\terror alone ({e})")
    all_classifiers = build_classifiers()
    for classifiers in all_classifiers:
        sc = stacked_classifier()
        try:
            predicted = build_stacked_classifier(classifiers, sc)
            accuracy = accuracy_score(test_labels, predicted)
            if print_results: print(f"\t{accuracy} with {classifiers}")
            results.append((accuracy, f"{sc} with {classifiers}"))
        except Exception as e:
            if print_results: print(f"\terror with {classifiers} ({e})")
    if print_results: print()
if print_results: print()

results.sort(reverse=True, key=lambda r: r[0])
printed = 0, 0.0
print("Best results:")
# Print at least 3 (ignoring non-stacked results), more if there's a tie
for result in results:
    if printed[0] >= 3 and printed[1] != result[0]: break
    if "alone" in result[1]:
        print(f"[{result[1]} ({result[0]})]")
    else:
        print(f"{result[1]} ({result[0]})")
        printed = printed[0] + 1, result[0]

Best results:
[KNeighborsClassifier() alone (1.0)]
[DecisionTreeClassifier() alone (1.0)]
GaussianNB() with (LinearRegression(), KNeighborsClassifier(), SVC()) (1.0)
GaussianNB() with (KNeighborsClassifier(), SVC(), DecisionTreeClassifier()) (1.0)
GaussianNB() with (SVC(), DecisionTreeClassifier(), GaussianNB()) (1.0)
GaussianNB() with (DecisionTreeClassifier(), GaussianNB(), QuadraticDiscriminantAnalysis()) (1.0)


## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [1450]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_accuracy_vector(predicted, labels):
    result = []
    for i in range(len(predicted)):
        if predicted[i] == labels[i]:
            result.append(0)
        else:
            result.append(1)
    return result

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [1451]:
def set_new_weights(model):
    # \frac{1 + I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1 + I(y_{n}\neq h_{t}(x_{n})}
	I = np.array(calculate_accuracy_vector(model.predict(test_set), test_labels))
	return (1 + I) / sum(1 + I)

Train the classifier with the code below:

In [1452]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00129702 0.00064851 0.00064851 0.00129702 0.00064851 0.00064851
 0.00064851 0.00064851 0.00064851 0.00064851 0.00129702 0.00129702
 0.00129702 0.00064851 0.00064851 0.00064851 0.00064851 0.00129702
 0.00064851 0.00129702 0.00064851 0.00064851 0.00064851 0.00129702
 0.00064851 0.00064851 0.00064851 0.00129702 0.00064851 0.00129702
 0.00129702 0.00129702 0.00064851 0.00129702 0.00064851 0.00064851
 0.00064851 0.00129702 0.00129702 0.00129702 0.00064851 0.00064851
 0.00129702 0.00129702 0.00129702 0.00064851 0.00129702 0.00064851
 0.00129702 0.00129702 0.00064851 0.00129702 0.00129702 0.00129702
 0.00129702 0.00129702 0.00064851 0.00129702 0.00129702 0.00064851
 0.00064851 0.00064851 0.00064851 0.00064851 0.00064851 0.00064851
 0.00129702 0.00064851 0.00129702 0.00064851 0.00064851 0.00064851
 0.00129702 0.00064851 0.00064851 0.00064851 0.00064851 0.00129702
 0.00064851 0.00129702 0.00064851 0.00129702 0.00129702 0.00064851
 0.00064851 0.00129702 0.00129702 0.00064851 0.00064851 0.0012

Set the validation data set:

In [1453]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [1454]:
def get_prediction(xs):
    res = []
    for x in xs:
        predictions = [classifier.predict(x.reshape(1, -1)).item() for classifier in classifiers]
        res.append(max(
            set(predictions),
            key=list(predictions).count
        ))
    return res

Test it:

In [1455]:
for _ in range(5):
    validate_x, validate_label = generate_data(1, dimension, labels)
    prediction = get_prediction(validate_x)[0]

    print(f"{prediction} (expected {validate_label.item()})")

1 (expected 1)
1 (expected 1)
0 (expected 0)
0 (expected 1)
1 (expected 0)
